# Session 2.1 — Streamlit Foundations & State Management

## What We're Building Today

A working Streamlit chat application that:
- Talks to Claude via the API
- Maintains conversation history across interactions
- Has a sidebar with chat history navigation
- Supports branding customization

## Learning Objectives

1. **Explain** Streamlit's execution model and why `st.session_state` is essential
2. **Build** a stateful chat app using `st.chat_message`, `st.chat_input`, and `st.session_state`
3. **Integrate** the Claude API into a Streamlit chat interface
4. **Use** Claude Code to customize your app's appearance

## Important

This notebook covers the **lecture demos**. The main artifact for today is `app/main.py` — your Streamlit chat application. After break, you'll work directly in that file.

In [ ]:
# Path setup — run this cell FIRST
# Notebooks run from notebooks/2_1/ but pipeline modules are at the project root
import sys
from pathlib import Path

_PROJECT_ROOT = Path.cwd().parent.parent
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

# Load environment variables from project root
from dotenv import load_dotenv
load_dotenv(_PROJECT_ROOT / ".env")

print(f"Project root: {_PROJECT_ROOT}")
print(f"Path configured: pipeline imports will work")

In [ ]:
# Verify Streamlit is installed
import streamlit as st
print(f"Streamlit version: {st.__version__}")
print("Streamlit is installed and ready!")
print()
print("To run a Streamlit app: streamlit run app/main.py")

In [ ]:
# Verify Claude API access
from dotenv import load_dotenv
load_dotenv()

import anthropic
client = anthropic.Anthropic()
response = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=50,
    messages=[{"role": "user", "content": "Say 'API connected!' in exactly 2 words."}]
)
print(response.content[0].text)

### Setup Checklist

- [ ] Streamlit installed
- [ ] Claude API connected
- [ ] Ready to explore Streamlit concepts

## The Streamlit Execution Model

Streamlit's key behavior: **every user interaction reruns the entire Python script from top to bottom.**

This means:
- Variables reset on every interaction
- Objects are recreated
- Without persistence, your app has "amnesia"

The demos below show this in action. **These are meant to be run as Streamlit apps, not in this notebook.** Copy each to a `.py` file and run with `streamlit run filename.py`.

In [ ]:
# Save this as demo_execution.py and run with: streamlit run demo_execution.py
#
# Watch what happens to `count` when you interact with the app!

DEMO_CODE = """
import streamlit as st

st.title("Execution Model Demo")

st.write("This line runs every time.")

name = st.text_input("Enter your name:")
st.write(f"Hello, {name}!")

count = 0
count += 1
st.write(f"Count: {count}")
"""

print("Copy to demo_execution.py and run with: streamlit run demo_execution.py")
print()
print(DEMO_CODE)

In [ ]:
# Save this as demo_state.py and run with: streamlit run demo_state.py
#
# Now count PERSISTS across interactions!

DEMO_CODE = """
import streamlit as st

st.title("Session State Demo")

if "count" not in st.session_state:
    st.session_state.count = 0

if st.button("Increment"):
    st.session_state.count += 1

st.write(f"Count: {st.session_state.count}")
"""

print("Copy to demo_state.py and run with: streamlit run demo_state.py")
print()
print(DEMO_CODE)

### The Initialization Pattern

This pattern is the key to Streamlit persistence:

```python
if "key" not in st.session_state:
    st.session_state.key = initial_value
```

**Rules:**
- `st.session_state` persists across reruns within a session
- Does NOT survive page refresh or server restart
- Each browser tab gets its own isolated session state
- You will write this pattern many times today

## Chat Components

Streamlit provides two built-in components for chat interfaces:

| Component | Purpose |
|-----------|--------|
| `st.chat_message("role")` | Display styled message bubbles (user/assistant) |
| `st.chat_input("prompt")` | Input bar pinned to bottom of page |

Together with `st.session_state`, these give you a complete chat interface.

In [ ]:
# The Chat Loop Pattern — the foundation of every Streamlit chat app
#
# Save to demo_chat.py and run with: streamlit run demo_chat.py

DEMO_CODE = """
import streamlit as st

st.title("Chat Loop Demo")

# Part 1: Initialize message history
if "messages" not in st.session_state:
    st.session_state.messages = []

# Part 2: Display all previous messages
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# Part 3: Handle new input
if prompt := st.chat_input("Ask a question..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    reply = f"You said: {prompt}"
    with st.chat_message("assistant"):
        st.markdown(reply)
    st.session_state.messages.append({"role": "assistant", "content": reply})
"""

print("Copy to demo_chat.py and run with: streamlit run demo_chat.py")
print()
print(DEMO_CODE)

### The Chat Loop Pattern

```text
THE CHAT LOOP (runs on every interaction)

┌──────────────────────────────────┐
│  1. Initialize session state     │ ← Only on first run
│     (if not already set)         │
├──────────────────────────────────┤
│  2. Display all stored messages  │ ← Renders full history
│     (loop through history)       │
├──────────────────────────────────┤
│  3. Handle new input             │ ← Only if user typed
│     - Add user msg to history    │
│     - Generate response          │
│     - Add response to history    │
└──────────────────────────────────┘
```

**Key insight:** The message format `{"role": "user", "content": "..."}` is the same format the Claude API expects. Store messages this way → pass them directly to Claude.

## Layout Components

| Component | Purpose | Where We Use It |
|-----------|---------|----------------|
| `st.sidebar` | Persistent side panel | Chat history, controls |
| `st.expander` | Collapsible section | Source citations, debug info |
| `st.rerun()` | Force immediate script rerun | After state changes (clear chat) |

The sidebar is where your chat history and controls live. The main area is for the conversation.

In [ ]:
# Layout reference — these components are in your starter template

LAYOUT_CODE = """
with st.sidebar:
    st.title("Controls")
    if st.button("Clear Chat"):
        st.session_state.messages = []
        st.rerun()

with st.expander("Show details"):
    st.write("Hidden until clicked.")
"""

print(LAYOUT_CODE)

## Looking Ahead: The Context Window Problem

Right now, we pass ALL messages to Claude:

```python
messages=st.session_state.messages  # ALL of them
```

5 messages? Fine. 20 messages? Fine. 200 messages? **Problem.**

Claude has a context window limit. Long conversations will eventually:
- Hit the token limit (error)
- Get expensive (more tokens = more cost)
- Slow down (more tokens = more latency)

**Session 2.2 solves this** with context window management: sliding windows, truncation, and summarization.

For today: conversations stay short. It just works.

---

## Guided Build: Chat Application

**After break**, you'll work in `app/main.py` directly — not in this notebook.

The starter template is ~80% complete. You implement two steps and trace one:

| Step | What You Build | Lines |
|------|---------------|-------|
| Step 1 | Session state initialization | ~6 lines |
| Step 4 | Trace the Pipeline (notebook exercise) | reading |
| Step 5 | Chat input handler | ~12 lines |

Steps 2 (sidebar) and 3 (display) are provided by the instructor.

Students implement Steps 1 and 5, and trace `app/rag.py` in the notebook (Step 4).

Run with: `streamlit run app/main.py`

### Step 1: Session State Initialization

Initialize three keys in `st.session_state`:

```python
# messages — current conversation's message list
# conversations — dict of all conversations (chat_id → messages)
# current_chat — ID of the active conversation
```

Use the guard pattern for each key.

### Steps 2 & 3: Sidebar and Display (PROVIDED)

These steps are **already implemented** in `app/main.py` — do not modify them.

**Step 2: Sidebar — Chat History**
- "+ New Chat" button creates new conversations
- Sidebar lists all conversations (labeled by first user message)
- Clicking a conversation switches to it
- "Clear Chat" empties the current conversation

**Step 3: Display Chat History**
- Shows a welcome message when no messages exist
- Loops through `st.session_state.messages` and renders each with `st.chat_message()`

**Step 4: Source Display Helper (PROVIDED)**
- `display_sources()` renders retrieved sources in a collapsible expander
- Called from your Step 5 implementation

> Open `app/main.py` and read Steps 2, 3, and 4. Notice how they use `st.session_state` — the same keys you initialized in Step 1.

In [ ]:
# Step 1: Session State — implement in app/main.py

# TODO: Initialize st.session_state.messages as an empty list
# Use the guard pattern: if "key" not in st.session_state

# TODO: Initialize st.session_state.conversations as an empty dict

# TODO: Initialize st.session_state.current_chat as None

print("Implement these in app/main.py, then run: streamlit run app/main.py")

### Step 4: Trace the Pipeline

The RAG pipeline from AI-2 is already wired into your app via `app/rag.py`. You do not need to implement the API call yourself — it is handled for you.

**Open `app/rag.py` and read through the flow.** Key things to notice:

- `get_response(question, messages)` is the single entry point
- It returns a `ChatResponse` object with:
  - `.answer` — the generated text from Claude
  - `.sources` — list of source citations used in the response
- Internally, it retrieves relevant chunks, builds a grounded prompt, and calls Claude

**Your job:** Understand the contract. You call `get_response()`, you get back an answer and sources. The retrieval, prompting, and generation are encapsulated.

Below, test the retrieval layer directly to see what chunks come back for different queries.

In [ ]:
from pipeline.retrieval.naive import naive_retrieve

# Test retrieval with different queries
results = naive_retrieve("What is the vacation policy?", top_k=5)
for r in results:
    print(f"[{r['score']:.3f}] {r['metadata']['source']}: {r['text'][:80]}...")

### Step 5: Chat Input Handler

Implement the chat handler in `app/main.py`. Use `get_response()` from `app/rag.py`:

```python
from app.rag import get_response, display_sources

if prompt := st.chat_input("Ask a question..."):
    # a. Append user message to st.session_state.messages
    # b. Display user message with st.chat_message("user")
    # c. Call get_response() to generate a reply:
    #    response = get_response(prompt, st.session_state.messages)
    # d. Display assistant message with st.chat_message("assistant"):
    #    response.answer for the text
    # e. Display sources:
    #    display_sources(response.sources)
    # f. Append assistant message to st.session_state.messages
    # g. Save: st.session_state.conversations[current_chat] = messages.copy()
```

**Pattern:**
```python
response = get_response(prompt, st.session_state.messages)
response.answer        # the generated text
display_sources(response.sources)  # renders citations in an expander
```

### Verify Your Implementation

After implementing Steps 1 and 5, run your app and test:

```bash
uv run streamlit run app/main.py
```

**Checklist — confirm each of these works:**

- [ ] App launches without errors
- [ ] Welcome message appears: "Hello! Ask me anything about Northbrook Partners."
- [ ] Chat input bar visible at bottom of page
- [ ] Type a question (e.g., "What is the vacation policy?") — get a grounded answer
- [ ] "Sources" expander appears below the response — click to expand and see citations
- [ ] Send a second message — both messages persist in the conversation
- [ ] Click "+ New Chat" — chat clears, welcome message returns
- [ ] Sidebar shows your old conversation (labeled with first ~30 characters)
- [ ] Click the old conversation — messages are restored
- [ ] "Messages" counter in sidebar updates correctly

> **If something isn't working:** Check the terminal where Streamlit is running for error messages. Common issues: missing `st.session_state` initialization (Step 1), `get_response` not called correctly (Step 5), or `.env` missing API keys.

---

## Claude Code: Customizing Your App

Claude Code is a command-line tool that connects Claude to your codebase. You describe changes in natural language, and it edits your files.

### Your Customization Surface

| File | What It Controls | Type |
|------|-----------------|------|
| `.streamlit/config.toml` | Colors, fonts, theme | Config (no code) |
| `.streamlit/static/` | Logo, images | Assets (no code) |
| `student_config.yaml` | App name, tagline, welcome message | Config (no code) |
| `app/branding.py` | Logo placement, CSS, favicon | Python |

**These files are yours.** The instructor will never push changes to them.

In [ ]:
# View your current student config
import yaml

with open("student_config.yaml") as f:
    config = yaml.safe_load(f)

for key, value in config.items():
    print(f"{key}: {value}")

---

## Lab 1: Streamlit Application Customization

**Assigned:** Today | **Due:** Session 3.1 | **Weight:** 15%

### Rubric (100 points)

| Criterion | Points | What to Do |
|-----------|--------|----------|
| Color Theme | 20 | Custom colors in `.streamlit/config.toml` |
| Logo & Branding | 20 | Custom logo + favicon |
| App Identity | 20 | App name + tagline in `student_config.yaml` |
| Welcome Message | 20 | Custom welcome message |
| Overall Polish | 20 | Cohesive, intentional, not broken |

### Files You Own
- `.streamlit/config.toml`
- `.streamlit/static/`
- `student_config.yaml`
- `app/branding.py`

### Files You Do NOT Touch
- `app/main.py` — instructor-managed
- `app/rag.py` — instructor-managed
- `pipeline/` — instructor-managed

**Out of scope:** Pipeline behavior changes earn zero points. Lab 1 assesses GUI/UX customization only.

---

## Questions to Leave With

> **On History:** "What happens when a conversation goes on for 50 messages? 100? Does your context window hold?"

> **On Integration:** "You have a chat UI and you have a retrieval pipeline. How do you connect them? Where in the code does retrieval happen?"

> **On Customization:** "What makes an AI application feel professional and trustworthy? How does branding affect user confidence?"

## Next Session: 2.2 — Context Management & Query Rewriting

- Context window management for long conversations
- Streaming responses for perceived speed
- Query rewriting for better follow-up questions
- Token budget enforcement and cost tracking

**Bring your working chat app from today!**

**For Lab 1:** Start customizing tonight. Use Claude Code. Make it yours.